# Chapitre 6 · Le neurone et le réseau (solutions des exercices)

Ce notebook contient **uniquement les réponses aux trois exercices** du notebook
du chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le minimum repris de la leçon pour que tout s'exécute ici de façon autonome :
le carnet de Sètondji et la fonction `accuracy`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

# Le carnet de Setondji : 400 courses, regle cachee a deux zones (lecon, section 3).
n_courses = 400
destinations = torch.rand(n_courses, 2)
destinations[:, 0] = destinations[:, 0] * 12.0 - 8.0   # est-ouest : -8 .. +4 km
destinations[:, 1] = destinations[:, 1] * 8.0 - 4.0    # nord-sud  : -4 .. +4 km
dist_marche = (destinations ** 2).sum(dim=1).sqrt()
dist_aeroport = ((destinations[:, 0] + 5.5) ** 2 + (destinations[:, 1] + 2.0) ** 2).sqrt()
cibles = ((dist_marche < 2.6) | (dist_aeroport < 2.2)).long()   # 1 = rentable, 0 = a perte

def accuracy(modele):
    """Le taux de bonnes reponses sur les 400 courses du carnet."""
    modele.eval()
    with torch.no_grad():
        predictions = modele(destinations).argmax(dim=1)
    modele.train()
    return (predictions == cibles).float().mean().item()

print(f"carnet pret : {tuple(destinations.shape)}, {cibles.sum().item()} courses rentables sur {n_courses}")

### Exercice 1 · Une couche, de tes mains — niveau ●

Écris le forward d'une couche : le produit matriciel du chapitre 2 (`@`), la
transposée de `W` pour emboîter les dimensions, puis `+ b`. Une seule ligne de
calcul. Rappel des shapes : `entrees (n, e)`, `W (s, e)` (une ligne par
neurone), `b (s,)`, sortie `(n, s)`.

In [ ]:
def forward_couche(entrees, W, b):
    """La couche : chaque ligne de W est la recette d'un neurone.

    entrees : (n, e)   W : (s, e)   b : (s,)   ->   sortie : (n, s)
    """
    return entrees @ W.T + b


# Demonstration sur des valeurs simples, verifiables de tete.
entrees_demo = torch.tensor([[2.0, 3.0],
                             [1.0, -1.0]])
W_demo = torch.tensor([[1.0, 0.0],     # neurone 1 : recopie x1
                       [0.0, 1.0],     # neurone 2 : recopie x2
                       [1.0, 1.0],     # neurone 3 : x1 + x2
                       [0.5, -0.5]])   # neurone 4 : (x1 - x2) / 2
b_demo = torch.tensor([0.0, 0.0, 1.0, 2.0])

sortie_demo = forward_couche(entrees_demo, W_demo, b_demo)
print(f"shape : {tuple(entrees_demo.shape)} @ {tuple(W_demo.T.shape)} + {tuple(b_demo.shape)} -> {tuple(sortie_demo.shape)}")
print(sortie_demo)


In [ ]:
# Validation de l'exercice 1 : les quatre recettes, calculees a la main.
attendu = torch.tensor([[2.0, 3.0, 6.0, 1.5],
                        [1.0, -1.0, 1.0, 3.0]])
assert sortie_demo is not None, "forward_couche ne renvoie rien : il manque le return ?"
assert tuple(sortie_demo.shape) == (2, 4), f"shape {tuple(sortie_demo.shape)} au lieu de (2, 4) : W est-elle bien transposee ?"
assert torch.allclose(sortie_demo, attendu), "les valeurs ne collent pas : verifie entrees @ W.T + b"
print("Exercice 1 valide : ta couche calcule juste.")


### Exercice 2 · La boucle d'entraînement en cinq lignes — niveau ●●

Le refrain du livre, de tête cette fois : forward, loss, `zero_grad()`,
`backward()`, `step()`, dans cet ordre. Repars d'un étage linéaire tout neuf et
entraîne-le sur le carnet : tu dois retomber exactement sur le plafond des 63 %
de la leçon.

In [ ]:
torch.manual_seed(1)
modele_exercice = nn.Linear(2, 2)
optimiseur = torch.optim.SGD(modele_exercice.parameters(), lr=0.2)

for etape in range(10001):
    logits = modele_exercice(destinations)          # 1. forward : (400, 2) -> (400, 2)
    loss = F.cross_entropy(logits, cibles)          # 2. mesurer l'erreur (chapitre 5)
    optimiseur.zero_grad()                          # 3. remettre les compteurs a zero
    loss.backward()                                 # 4. le retour : tous les gradients
    optimiseur.step()                               # 5. le pas de descente (chapitre 3)
    if etape % 2000 == 0:
        print(f"etape {etape:5d} | loss = {loss.item():.4f}")

In [ ]:
# Validation de l'exercice 2 : la loss a fondu, le modele fait mieux que le hasard.
loss_exercice = F.cross_entropy(modele_exercice(destinations), cibles).item()
acc_exercice = accuracy(modele_exercice)
print(f"loss finale : {loss_exercice:.4f} | accuracy : {acc_exercice:.3f}")

assert loss_exercice < 0.65, "la loss n'a pas assez fondu : les 5 lignes sont-elles dans le bon ordre ?"
assert 0.55 < acc_exercice < 0.72, "l'accuracy attendue est ~0.63 : relis la boucle"
print("Exercice 2 valide : ta boucle entraine. Meme plafond de 63 % que la lecon, comme prevu.")

### Exercice 3 · ReLU et le forward d'un MLP — niveau ●●●

Écris `relu_maison` (`max(0, x)`, appliqué élément par élément : le pli), puis
le forward complet d'un MLP : couche, pli, couche. Réutilise ta `forward_couche`
de l'exercice 1.

In [ ]:
def relu_maison(t):
    """max(0, x), applique element par element : le pli."""
    return torch.clamp(t, min=0.0)


def forward_mlp(entrees, W1, b1, W2, b2):
    """Couche -> pli -> couche : le plus petit MLP du monde."""
    cachee = relu_maison(forward_couche(entrees, W1, b1))
    return forward_couche(cachee, W2, b2)


# Demonstration, verifiable a la main.
e = torch.tensor([[1.0, 2.0], [-2.0, 0.5]])
W1_demo = torch.tensor([[1.0, 0.0], [-1.0, 1.0]])
b1_demo = torch.tensor([0.0, 1.0])
W2_demo = torch.tensor([[2.0, -1.0]])
b2_demo = torch.tensor([0.5])
sortie_mlp = forward_mlp(e, W1_demo, b1_demo, W2_demo, b2_demo)
print(sortie_mlp)


In [ ]:
# Validation de l'exercice 3, pas a pas puis de bout en bout.
assert torch.equal(relu_maison(torch.tensor([-3.0, 0.0, 2.5])), torch.tensor([0.0, 0.0, 2.5])), \
    "relu_maison doit ecraser les negatifs a zero et laisser passer le reste"

# A la main : pre-activation [[1, 2], [-2, 3.5]] -> relu [[1, 2], [0, 3.5]]
# -> sortie [2*1 - 1*2 + 0.5, 2*0 - 1*3.5 + 0.5] = [0.5, -3.0]
attendu_mlp = torch.tensor([[0.5], [-3.0]])
assert sortie_mlp is not None and torch.allclose(sortie_mlp, attendu_mlp), \
    "le forward du MLP ne colle pas : couche -> pli -> couche, dans cet ordre"
print("Exercice 3 valide : tu sais ecrire un MLP a la main.")
